In [3]:
import pandas as pd
import numpy as np
import glob
import hashlib
from pathlib import Path
from datetime import datetime

print("Libraries loaded")

Libraries loaded


In [4]:

file_2000_2012 = INPUT_DIR / "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv"
file_2012_2014 = INPUT_DIR / "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv"
file_2015_2016 = INPUT_DIR / "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv"

df_2000_2012 = pd.read_csv(file_2000_2012)
df_2012_2014 = pd.read_csv(file_2012_2014)
df_2015_2016 = pd.read_csv(file_2015_2016)

print("2000-Feb 2012:", df_2000_2012.shape)
print("Mar 2012-Dec 2014:", df_2012_2014.shape)
print("Jan 2015-Dec 2016:", df_2015_2016.shape)

2000-Feb 2012: (369651, 10)
Mar 2012-Dec 2014: (52203, 10)
Jan 2015-Dec 2016: (37153, 11)


In [5]:
print("2000-Feb 2012 columns:")
print(df_2000_2012.columns.tolist())

print("\nMar 2012-Dec 2014 columns:")
print(df_2012_2014.columns.tolist())

print("\nJan 2015-Dec 2016 columns:")
print(df_2015_2016.columns.tolist())

2000-Feb 2012 columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

Mar 2012-Dec 2014 columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

Jan 2015-Dec 2016 columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']


In [6]:
# Combine all three required datasets into a single master dataset.

master_data = pd.concat(
    [
        df_2000_2012,
        df_2012_2014,
        df_2015_2016
    ],
    ignore_index=True,
    sort=False
)

print("Master dataset shape:", master_data.shape)
print("\nMaster dataset columns:")
print(master_data.columns.tolist())

Master dataset shape: (459007, 11)

Master dataset columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


In [7]:
# Create January 2012 authoritative dataset

jan_2012 = df_2000_2012[
    df_2000_2012["month"].str.startswith("2012-01")
].copy()

print("January 2012 rows:", len(jan_2012))
print("January 2012 date range:", jan_2012["month"].min(), "to", jan_2012["month"].max())

January 2012 rows: 1559
January 2012 date range: 2012-01 to 2012-01


In [8]:
# Create reference value sets from January 2012

reference_town = set(jan_2012["town"].dropna().unique())
reference_flat_type = set(jan_2012["flat_type"].dropna().unique())
reference_flat_model = set(jan_2012["flat_model"].dropna().unique())
reference_storey_range = set(jan_2012["storey_range"].dropna().unique())

print("Reference Town values:", len(reference_town))
print("Reference Flat Type values:", len(reference_flat_type))
print("Reference Flat Model values:", len(reference_flat_model))
print("Reference Storey Range values:", len(reference_storey_range))

Reference Town values: 26
Reference Flat Type values: 7
Reference Flat Model values: 13
Reference Storey Range values: 12


In [9]:

master_data["valid_town"] = master_data["town"].isin(reference_town)
master_data["valid_flat_type"] = master_data["flat_type"].isin(reference_flat_type)
master_data["valid_flat_model"] = master_data["flat_model"].isin(reference_flat_model)
master_data["valid_storey_range"] = master_data["storey_range"].isin(reference_storey_range)

print("Invalid Town:", (~master_data["valid_town"]).sum())
print("Invalid Flat Type:", (~master_data["valid_flat_type"]).sum())
print("Invalid Flat Model:", (~master_data["valid_flat_model"]).sum())
print("Invalid Storey Range:", (~master_data["valid_storey_range"]).sum())

Invalid Town: 0
Invalid Flat Type: 0
Invalid Flat Model: 620
Invalid Storey Range: 7009


In [10]:

master_data["month_date"] = pd.to_datetime(
    master_data["month"],
    format="%Y-%m",
    errors="coerce"
)

master_data["valid_month"] = master_data["month_date"].between(
    "2012-01-01",
    "2016-12-01"
)

print("Invalid Month:", (~master_data["valid_month"]).sum())

Invalid Month: 366463


In [12]:
master_data["month_date"] = pd.to_datetime(
    master_data["month"],
    format="%Y-%m",
    errors="coerce"
)

master_data = master_data[
    master_data["month_date"].between("2012-01-01", "2016-12-31")
].copy()

print("Rows after date filtering:", len(master_data))
print("Date range:", master_data["month"].min(), "to", master_data["month"].max())

Rows after date filtering: 92544
Date range: 2012-01 to 2016-12


In [13]:

master_data["validation_status"] = np.where(
    master_data[
        [
            "valid_town",
            "valid_flat_type",
            "valid_flat_model",
            "valid_storey_range",
            "valid_month"
        ]
    ].all(axis=1),
    "Valid",
    "Invalid"
)

print(master_data["validation_status"].value_counts())

Valid      85133
Invalid     7411
Name: validation_status, dtype: int64


In [14]:
#check fail
validation_columns = [
    "valid_town",
    "valid_flat_type",
    "valid_flat_model",
    "valid_storey_range",
    "valid_month"
]

for column in validation_columns:
    print(column, ":", (~master_data[column]).sum())

valid_town : 0
valid_flat_type : 0
valid_flat_model : 492
valid_storey_range : 6975
valid_month : 0


In [16]:
#reconcile valid and invalid

def get_validation_reason(row):
    reasons = []

    if not row["valid_town"]:
        reasons.append("Invalid town")

    if not row["valid_flat_type"]:
        reasons.append("Invalid flat_type")

    if not row["valid_flat_model"]:
        reasons.append("Invalid flat_model")

    if not row["valid_storey_range"]:
        reasons.append("Invalid storey_range")

    if not row["valid_month"]:
        reasons.append("Invalid month")

    return ", ".join(reasons) if reasons else "Valid"


master_data["validation_reason"] = master_data.apply(
    get_validation_reason,
    axis=1
)

print(master_data["validation_reason"].value_counts())

Valid                                       85133
Invalid storey_range                         6919
Invalid flat_model                            436
Invalid flat_model, Invalid storey_range       56
Name: validation_reason, dtype: int64


In [17]:
# Split the data into valid and invalid records

cleaned_data = master_data[
    master_data["validation_status"] == "Valid"
].copy()

invalid_data = master_data[
    master_data["validation_status"] == "Invalid"
].copy()

print("Cleaned rows:", len(cleaned_data))
print("Invalid rows:", len(invalid_data))

Cleaned rows: 85133
Invalid rows: 7411


In [18]:
# Save the cleaned and invalid records

cleaned_data.to_csv(
    OUTPUT_DIR / "hdb_resale_cleaned.csv",
    index=False
)

invalid_data.to_csv(
    OUTPUT_DIR / "hdb_resale_invalid.csv",
    index=False
)

print("Cleaned file saved.")
print("Invalid file saved.")

Cleaned file saved.
Invalid file saved.


In [19]:
#  duplicate records

duplicate_columns = [
    column for column in cleaned_data.columns
    if column != "resale_price"
]

duplicate_count = cleaned_data.duplicated(
    subset=duplicate_columns,
    keep=False
).sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 2738


In [22]:
# duplicate records

duplicates = cleaned_data[
    cleaned_data.duplicated(
        subset=duplicate_columns,
        keep=False
    )
].sort_values(duplicate_columns)

duplicates.head(10)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,valid_town,valid_flat_type,valid_flat_model,valid_storey_range,month_date,valid_month,validation_status,validation_reason
366493,2012-01,ANG MO KIO,3 ROOM,256,ANG MO KIO AVE 4,10 TO 12,73.0,New Generation,1977,353000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366494,2012-01,ANG MO KIO,3 ROOM,256,ANG MO KIO AVE 4,10 TO 12,73.0,New Generation,1977,340000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366507,2012-01,ANG MO KIO,3 ROOM,506,ANG MO KIO AVE 8,07 TO 09,68.0,New Generation,1980,342000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366508,2012-01,ANG MO KIO,3 ROOM,506,ANG MO KIO AVE 8,07 TO 09,68.0,New Generation,1980,334000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366478,2012-01,ANG MO KIO,3 ROOM,558,ANG MO KIO AVE 10,10 TO 12,67.0,New Generation,1980,336000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366487,2012-01,ANG MO KIO,3 ROOM,558,ANG MO KIO AVE 10,10 TO 12,67.0,New Generation,1980,344000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366505,2012-01,ANG MO KIO,3 ROOM,633,ANG MO KIO AVE 6,10 TO 12,67.0,New Generation,1985,366000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366506,2012-01,ANG MO KIO,3 ROOM,633,ANG MO KIO AVE 6,10 TO 12,67.0,New Generation,1985,350000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366571,2012-01,BEDOK,3 ROOM,138,BEDOK NTH ST 2,10 TO 12,67.0,New Generation,1978,344000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid
366572,2012-01,BEDOK,3 ROOM,138,BEDOK NTH ST 2,10 TO 12,67.0,New Generation,1978,316000.0,NaN,True,True,True,True,2012-01-01,True,Valid,Valid


In [23]:
duplicate_columns = [
    "month", "town", "flat_type", "block", "street_name",
    "storey_range", "floor_area_sqm", "flat_model",
    "lease_commence_date"
]

duplicates = cleaned_data[
    cleaned_data.duplicated(subset=duplicate_columns, keep=False)
]

print("Duplicate records:", len(duplicates))

Duplicate records: 2748


In [25]:


cleaned_data = cleaned_data.sort_values(
    "resale_price",
    ascending=False
)

cleaned_data = cleaned_data.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).copy()

print("Rows after removing duplicates:", len(cleaned_data))

Rows after removing duplicates: 83745


In [27]:

print("Data count summary")
print("------------------")
print("Records after date filter:", len(master_data))
print("Valid records:", valid_before_duplicates)
print("Invalid records:", len(invalid_data))
print("Duplicate records removed:", valid_before_duplicates - len(cleaned_data))
print("Records after duplicate handling:", len(cleaned_data))

Data count summary
------------------
Records after date filter: 92544
Valid records: 85133
Invalid records: 7411
Duplicate records removed: 1388
Records after duplicate handling: 83745


In [28]:

total_after_date_filter = len(master_data)
total_valid = (master_data["validation_status"] == "Valid").sum()
total_invalid = (master_data["validation_status"] == "Invalid").sum()
total_after_duplicates = len(cleaned_data)
duplicates_removed = total_valid - total_after_duplicates

In [29]:
remaining_duplicates = cleaned_data.duplicated(
    subset=duplicate_columns,
    keep=False
).sum()

remaining_duplicates

0

In [30]:

cleaned_data.to_csv(
    OUTPUT_DIR / "hdb_resale_cleaned.csv",
    index=False
)

In [31]:

saved_cleaned_data = pd.read_csv(
    OUTPUT_DIR / "hdb_resale_cleaned.csv"
)

len(saved_cleaned_data)

83745

In [33]:
duplicates.to_csv(
    OUTPUT_DIR / "hdb_resale_duplicates.csv",
    index=False
)

In [34]:
saved_duplicates = pd.read_csv(
    OUTPUT_DIR / "hdb_resale_duplicates.csv"
)

len(saved_duplicates)

2748

In [35]:
price_group = cleaned_data.groupby(
    ["month", "flat_type"]
)["resale_price"]

q1 = price_group.transform("quantile", 0.25)
q3 = price_group.transform("quantile", 0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

cleaned_data["price_anomaly"] = (
    (cleaned_data["resale_price"] < lower_bound) |
    (cleaned_data["resale_price"] > upper_bound)
)

In [36]:
anomaly_count = cleaned_data["price_anomaly"].sum()
anomaly_count

5434

In [37]:
anomalies = cleaned_data[cleaned_data["price_anomaly"]]
print("Anomalies detected:", len(anomalies))
anomalies.head()


Anomalies detected: 5434


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,valid_town,valid_flat_type,valid_flat_model,valid_storey_range,month_date,valid_month,validation_status,validation_reason,price_anomaly
458344,2016-12,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,1150000.0,54.0,True,True,True,True,2016-12-01,True,Valid,Valid,True
417838,2014-10,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.0,Maisonette,1987,1088888.0,NaN,True,True,True,True,2014-10-01,True,Valid,Valid,True
424967,2015-03,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,280.0,Terrace,1972,1060000.0,56.0,True,True,True,True,2015-03-01,True,Valid,Valid,True
425819,2015-04,BISHAN,EXECUTIVE,192,BISHAN ST 13,22 TO 24,149.0,Maisonette,1987,1050000.0,71.0,True,True,True,True,2015-04-01,True,Valid,Valid,True
451066,2016-08,BISHAN,EXECUTIVE,187,BISHAN ST 13,07 TO 09,153.0,Maisonette,1987,1050000.0,69.0,True,True,True,True,2016-08-01,True,Valid,Valid,True



Assuming an HDB lease of 99 years, calculate the remaining lease based on the lease commencement year and transaction date. The result is rounded down and expressed in years and months.

In [38]:
lease_start = pd.to_datetime(
    cleaned_data["lease_commence_date"].astype(str),
    format="%Y",
    errors="coerce"
)

transaction_date = cleaned_data["month_date"]

lease_end = lease_start + pd.DateOffset(years=99)

remaining_days = (lease_end - transaction_date).dt.days

remaining_years = remaining_days // 365
remaining_months = (remaining_days % 365) // 30

cleaned_data["calculated_remaining_lease"] = (
    remaining_years.astype("Int64").astype(str)
    + " years "
    + remaining_months.astype("Int64").astype(str)
    + " months"
)

In [39]:
cleaned_data[
    [
        "month",
        "lease_commence_date",
        "remaining_lease",
        "calculated_remaining_lease"
    ]
].head(10)

,month,lease_commence_date,remaining_lease,calculated_remaining_lease
458344,2016-12,1972,54.0,54 years 1 months
417838,2014-10,1987,NaN,71 years 3 months
424967,2015-03,1972,56.0,55 years 10 months
425819,2015-04,1987,71.0,70 years 9 months
451066,2016-08,1987,69.0,69 years 5 months
403701,2013-11,1987,NaN,72 years 2 months
394387,2013-04,1972,NaN,57 years 9 months
384739,2012-10,1987,NaN,73 years 3 months
451435,2016-08,2012,94.0,94 years 5 months
379883,2012-07,1995,NaN,81 years 6 months


In [40]:
average_price = cleaned_data.groupby(
    ["month", "town", "flat_type"]
)["resale_price"].transform("mean")

In [41]:
average_price = cleaned_data.groupby(
    ["month", "town", "flat_type"]
)["resale_price"].transform("mean")

block = cleaned_data["block"].str.replace(r"\D", "", regex=True).str[:3].str.zfill(3)
price = average_price.round().astype(int).astype(str).str[:2]
month = cleaned_data["month"].str[-2:]
town = cleaned_data["town"].str[0]

cleaned_data["Resale Identifier"] = (
    "S" + block + price + month + town
)

In [42]:
cleaned_data[
    ["month", "town", "flat_type", "block", "resale_price", "Resale Identifier"]
].head(10)

,month,town,flat_type,block,resale_price,Resale Identifier
458344,2016-12,KALLANG/WHAMPOA,3 ROOM,57,1150000.0,S0573612K
417838,2014-10,BISHAN,EXECUTIVE,194,1088888.0,S1949410B
424967,2015-03,KALLANG/WHAMPOA,3 ROOM,53,1060000.0,S0534103K
425819,2015-04,BISHAN,EXECUTIVE,192,1050000.0,S1929304B
451066,2016-08,BISHAN,EXECUTIVE,187,1050000.0,S1879108B
403701,2013-11,BISHAN,EXECUTIVE,190,1050000.0,S1909411B
394387,2013-04,KALLANG/WHAMPOA,3 ROOM,65,1020000.0,S0655104K
384739,2012-10,BISHAN,EXECUTIVE,194,1010000.0,S1948610B
451435,2016-08,CLEMENTI,5 ROOM,441A,1005000.0,S4417208C
379883,2012-07,QUEENSTOWN,EXECUTIVE,149,1000000.0,S1491007Q




Create a unique identifier using the block, average resale price, transaction month and town.

For example, `S0573612K`:

- `S` = required first character
- `057` = first 3 numeric digits from block `57`, padded with zero
- `36` = first 2 digits of the average resale price for the month, town and flat type
- `12` = transaction month
- `K` = first character of KALLANG/WHAMPOA

In [43]:
cleaned_data["Resale Identifier"].nunique() == len(cleaned_data)

False

In [44]:
duplicate_identifiers = cleaned_data[
    cleaned_data["Resale Identifier"].duplicated(keep=False)
]

len(duplicate_identifiers)

21665

In [45]:
duplicate_identifiers["Resale Identifier"].nunique()

9723

In [46]:
duplicate_identifiers[
    ["Resale Identifier", "month", "town", "flat_type", "block", "resale_price"]
].head(10)

,Resale Identifier,month,town,flat_type,block,resale_price
458344,S0573612K,2016-12,KALLANG/WHAMPOA,3 ROOM,57,1150000.0
445814,S1868805B,2016-05,BISHAN,EXECUTIVE,186,979000.0
446791,S0188805Q,2016-05,QUEENSTOWN,5 ROOM,18C,968000.0
399050,S0918607Q,2013-07,QUEENSTOWN,5 ROOM,91,961000.0
445813,S1878805B,2016-05,BISHAN,EXECUTIVE,187,955000.0
443196,S0188903Q,2016-03,QUEENSTOWN,5 ROOM,18D,955000.0
435387,S1277810B,2015-10,BUKIT MERAH,5 ROOM,127D,950000.0
415528,S0633708K,2014-08,KALLANG/WHAMPOA,3 ROOM,63,940000.0
395410,S1868805B,2013-05,BISHAN,EXECUTIVE,186,940000.0
399049,S0918607Q,2013-07,QUEENSTOWN,5 ROOM,91,935000.0


In [47]:
cleaned_data["Hashed Resale Identifier"] = cleaned_data["Resale Identifier"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

cleaned_data["Hashed Resale Identifier"].nunique() == cleaned_data["Resale Identifier"].nunique()

True

In [48]:
anomalies["town"].value_counts()

BUKIT MERAH        1382
QUEENSTOWN          997
KALLANG/WHAMPOA     465
TOA PAYOH           460
BISHAN              395
GEYLANG             313
CLEMENTI            272
MARINE PARADE       234
ANG MO KIO          211
CENTRAL AREA        199
BEDOK               180
BUKIT TIMAH         112
SERANGOON           111
SENGKANG             39
HOUGANG              21
BUKIT BATOK          17
JURONG WEST           8
TAMPINES              6
JURONG EAST           5
PASIR RIS             3
WOODLANDS             2
PUNGGOL               1
YISHUN                1
Name: town, dtype: int64

In [52]:
cleaned_data.columns.tolist()

['month',
 'town',
 'flat_type',
 'block',
 'street_name',
 'storey_range',
 'floor_area_sqm',
 'flat_model',
 'lease_commence_date',
 'resale_price',
 'remaining_lease',
 'price_anomaly',
 'calculated_remaining_lease',
 'Resale Identifier',
 'Hashed Resale Identifier']

In [53]:
cleaned_data["remaining_lease"] = cleaned_data["calculated_remaining_lease"]

# Drop the temporary calculated column
cleaned_data = cleaned_data.drop(columns=["calculated_remaining_lease"])


In [54]:
cleaned_data.columns.tolist()

['month',
 'town',
 'flat_type',
 'block',
 'street_name',
 'storey_range',
 'floor_area_sqm',
 'flat_model',
 'lease_commence_date',
 'resale_price',
 'remaining_lease',
 'price_anomaly',
 'Resale Identifier',
 'Hashed Resale Identifier']



Save the cleaned and validated records after duplicate handling and transformations.

In [55]:
cleaned_data.to_csv(
    OUTPUT_DIR / "hdb_resale_cleaned.csv",
    index=False
)

In [56]:
len(pd.read_csv(OUTPUT_DIR / "hdb_resale_cleaned.csv"))

83745

In [57]:
cleaned_data.to_csv(
    OUTPUT_DIR / "hdb_resale_final.csv",
    index=False
)

In [58]:
len(pd.read_csv(OUTPUT_DIR / "hdb_resale_final.csv"))

83745

In [59]:
final_data = pd.read_csv(
    OUTPUT_DIR / "hdb_resale_final.csv"
)

final_data.shape

(83745, 14)

In [60]:
len(pd.read_csv(OUTPUT_DIR / "hdb_resale_invalid.csv"))

7411

In [62]:
list(INPUT_DIR.glob("*.csv"))

[WindowsPath('../data/input/Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv'),
 WindowsPath('../data/input/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv'),
 WindowsPath('../data/input/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv'),
 WindowsPath('../data/input/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv'),
 WindowsPath('../data/input/Resale flat prices based on registration date from Jan-2017 onwards.csv')]

In [63]:
import shutil

raw_dir = OUTPUT_DIR / "raw"
raw_dir.mkdir(exist_ok=True)

for file in [file_2000_2012, file_2012_2014, file_2015_2016]:
    shutil.copy2(file, raw_dir / file.name)

Output Files

The processed data is organized into the five output groups required by the assessment: Raw, Cleaned, Transformed, Quarantined, and Hashed.


In [66]:
cleaned_dir = OUTPUT_DIR / "cleaned"
transformed_dir = OUTPUT_DIR / "transformed"
quarantined_dir = OUTPUT_DIR / "quarantined"
hashed_dir = OUTPUT_DIR / "hashed"

for folder in [
    cleaned_dir,
    transformed_dir,
    quarantined_dir,
    hashed_dir
]:
    folder.mkdir(exist_ok=True)

In [67]:
cleaned_output = cleaned_data.drop(
    columns=["Resale Identifier", "Hashed Resale Identifier"]
)

cleaned_output.to_csv(
    cleaned_dir / "hdb_resale_cleaned.csv",
    index=False
)

In [68]:
transformed_data = cleaned_data.drop(
    columns=["Hashed Resale Identifier"]
)

transformed_data.to_csv(
    transformed_dir / "hdb_resale_transformed.csv",
    index=False
)

In [69]:
quarantine_columns = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "resale_price"
]

invalid_quarantine = invalid_data[quarantine_columns].copy()
invalid_quarantine["quarantine_reason"] = "Validation failure"

duplicate_quarantine = duplicates[quarantine_columns].copy()
duplicate_quarantine["quarantine_reason"] = "Duplicate record"

anomaly_quarantine = anomalies[quarantine_columns].copy()
anomaly_quarantine["quarantine_reason"] = "Potential price anomaly"

quarantined_data = pd.concat(
    [
        invalid_quarantine,
        duplicate_quarantine,
        anomaly_quarantine
    ],
    ignore_index=True
)

quarantined_data.to_csv(
    quarantined_dir / "hdb_resale_quarantined.csv",
    index=False
)

In [70]:
len(quarantined_data)

15593

In [71]:
hashed_data = cleaned_data.copy()

hashed_data.to_csv(
    hashed_dir / "hdb_resale_hashed.csv",
    index=False
)

In [72]:
list(OUTPUT_DIR.rglob("*.csv"))

[WindowsPath('../data/output/hdb_resale_cleaned.csv'),
 WindowsPath('../data/output/hdb_resale_duplicates.csv'),
 WindowsPath('../data/output/hdb_resale_final.csv'),
 WindowsPath('../data/output/hdb_resale_invalid.csv'),
 WindowsPath('../data/output/cleaned/hdb_resale_cleaned.csv'),
 WindowsPath('../data/output/hashed/hdb_resale_hashed.csv'),
 WindowsPath('../data/output/quarantined/hdb_resale_quarantined.csv'),
 WindowsPath('../data/output/raw/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv'),
 WindowsPath('../data/output/raw/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv'),
 WindowsPath('../data/output/raw/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv'),
 WindowsPath('../data/output/transformed/hdb_resale_transformed.csv')]

In [64]:
list(raw_dir.glob("*.csv"))

[WindowsPath('../data/output/raw/Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv'),
 WindowsPath('../data/output/raw/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv'),
 WindowsPath('../data/output/raw/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv')]

In [65]:
for file in OUTPUT_DIR.rglob("*.csv"):
    print(file.relative_to(OUTPUT_DIR), len(pd.read_csv(file)))

hdb_resale_cleaned.csv 83745
hdb_resale_duplicates.csv 2748
hdb_resale_final.csv 83745
hdb_resale_invalid.csv 7411
raw\Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv 369651
raw\Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv 37153
raw\Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv 52203


ETL Summary / Results

Input period: January 2012 – December 2016 resale transactions
Records after date filtering: 92,544 rows
Valid records after validation: 85,133 rows
Invalid records: 7,411 rows flagged and stored separately (hdb_resale_invalid.csv)
Duplicate records identified: 2,748 rows involved in duplicate groups
Duplicates removed: 1,388 lower-priced duplicate records
Final cleaned records: 83,745 rows
Price anomalies flagged: 5,434 rows marked using the IQR rule and retained
Remaining lease calculation: Calculated using the assumed 99-year HDB lease and expressed in years and months
Resale Identifier: Generated using the required identifier format
SHA-256 hash: Added to create an irreversible hashed identifier

Output files:

  hdb_resale_cleaned.csv → 83,745 records
  hdb_resale_final.csv → 83,745 records
  hdb_resale_duplicates.csv → 2,748 records identified in duplicate groups
  hdb_resale_invalid.csv → 7,411 records
